# VendorCenter AI Assistant — Fine-Tuning Notebook

**Model**: `unsloth/Llama-3.2-3B-Instruct` (QLoRA 4-bit)  
**Output**: GGUF Q4_K_M for llama.cpp deployment on HuggingFace Spaces  
**Training data**: `vendorcenter_train.jsonl` (~1500+ examples from expanded generator)  
**Intents**: SERVICE_SEARCH, FAQ, GREETING, MY_BOOKINGS, AVAILABLE_SERVICES, COMPLAINT, RESCHEDULE, CANCEL_BOOKING, REFUND, VENDOR_INFO, LOCATION, UNKNOWN  

## Steps
1. Install deps (unsloth, trl, bitsandbytes, scikit-learn, matplotlib)
2. Upload training JSONL
3. Load, inspect and validate training data
4. Format into Llama-3.2 ChatML and split train/eval (90/10)
5. Load base model with 4-bit quantization
6. Apply LoRA adapters
7. Train with SFTTrainer (with eval)
8. Visualize training loss
9. Evaluate on held-out set (JSON validity, intent accuracy)
10. Multi-intent test suite
11. Merge & export GGUF Q4_K_M
12. Push to HuggingFace Hub or download

> **Runtime**: Use T4 GPU (free tier). Training ~1500 examples takes ~30-50 min.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl peft accelerate bitsandbytes datasets huggingface_hub
!pip install -q "torch>=2.1" --index-url https://download.pytorch.org/whl/cu121
!pip install -q scikit-learn matplotlib

In [ ]:
# Cell 2: Upload training data
# Option A: Upload manually via Colab's file browser (left sidebar → folder icon → upload)
# Option B: Use google.colab.files
from google.colab import files
import os

if not os.path.exists('vendorcenter_train.jsonl'):
    print('Upload vendorcenter_train.jsonl from model/training-data/')
    uploaded = files.upload()
    print(f'Uploaded: {list(uploaded.keys())}')
else:
    print('Training file already present.')

In [ ]:
# Cell 3: Load, inspect, and validate training data
import json
from collections import Counter

examples = []
invalid_rows = []
with open('vendorcenter_train.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if line.strip():
            try:
                row = json.loads(line)
                # Validate output is parseable JSON
                parsed_output = json.loads(row['output'])
                assert 'intent' in parsed_output, f'Row {i}: missing intent'
                assert 'message' in parsed_output, f'Row {i}: missing message'
                examples.append(row)
            except Exception as e:
                invalid_rows.append((i, str(e)))

print(f'Total valid examples: {len(examples)}')
print(f'Invalid rows: {len(invalid_rows)}')
if invalid_rows:
    for idx, err in invalid_rows[:5]:
        print(f'  Row {idx}: {err}')

# Intent distribution
intents = [json.loads(ex['output'])['intent'] for ex in examples]
intent_dist = Counter(intents)
print(f'\nIntent distribution ({len(intent_dist)} intents):')
for intent, count in intent_dist.most_common():
    pct = count / len(examples) * 100
    print(f'  {intent:25s} {count:4d}  ({pct:.1f}%)')

# Sample
print(f'\nSample:')
print(json.dumps(examples[0], indent=2, ensure_ascii=False)[:500])

In [ ]:
# Cell 4: Format into ChatML for Llama-3.2-Instruct + train/eval split
import random
random.seed(42)

def format_chatml(example):
    """Convert instruction/input/output to Llama-3.2-Instruct chat format."""
    return {
        'text': (
            f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
            f'{example["instruction"]}<|eot_id|>'
            f'<|start_header_id|>user<|end_header_id|>\n\n'
            f'{example["input"]}<|eot_id|>'
            f'<|start_header_id|>assistant<|end_header_id|>\n\n'
            f'{example["output"]}<|eot_id|>'
        ),
        'intent': json.loads(example['output'])['intent'],
        'raw_input': example['input'],
        'raw_output': example['output'],
    }

formatted = [format_chatml(ex) for ex in examples]

# Stratified-ish shuffle then split 90/10
random.shuffle(formatted)
split_idx = int(len(formatted) * 0.9)
train_formatted = formatted[:split_idx]
eval_formatted = formatted[split_idx:]

print(f'Train: {len(train_formatted)} | Eval: {len(eval_formatted)}')
print(f'\nTrain intent distribution:')
for intent, count in Counter(e['intent'] for e in train_formatted).most_common():
    print(f'  {intent}: {count}')
print(f'\nEval intent distribution:')
for intent, count in Counter(e['intent'] for e in eval_formatted).most_common():
    print(f'  {intent}: {count}')

print(f'\nSample (first 400 chars):\n{train_formatted[0]["text"][:400]}')

In [ ]:
# Cell 5: Create HuggingFace Datasets (train + eval)
from datasets import Dataset

train_dataset = Dataset.from_list([{'text': e['text']} for e in train_formatted])
eval_dataset = Dataset.from_list([{'text': e['text']} for e in eval_formatted])

print(f'Train dataset: {train_dataset}')
print(f'Eval dataset:  {eval_dataset}')
print(f'\nSample:\n{train_dataset[0]["text"][:300]}')

In [ ]:
# Cell 6: Load base model with 4-bit quantization
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters():,}')

In [ ]:
# Cell 7: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # LoRA rank
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=128,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# Print trainable params
model.print_trainable_parameters()

In [ ]:
# Cell 8: Configure SFTTrainer (with eval)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=True,  # pack short examples together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy='epoch',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='vendorcenter-lora',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
    ),
)

print('Trainer configured with train + eval. Ready to train.')

In [ ]:
# Cell 9: Train!
import time

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f'\nTraining complete in {elapsed/60:.1f} minutes')
print(f'Final loss: {stats.training_loss:.4f}')

In [ ]:
# Cell 10: Visualize training & eval loss
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_steps = [h['step'] for h in log_history if 'loss' in h]
train_losses = [h['loss'] for h in log_history if 'loss' in h]
eval_steps = [h['step'] for h in log_history if 'eval_loss' in h]
eval_losses = [h['eval_loss'] for h in log_history if 'eval_loss' in h]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label='Train Loss', alpha=0.8)
if eval_losses:
    plt.plot(eval_steps, eval_losses, 'ro-', label='Eval Loss', markersize=8)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('VendorCenter Fine-Tuning — Loss Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print(f'Train loss: {train_losses[0]:.4f} → {train_losses[-1]:.4f}')
if eval_losses:
    print(f'Eval loss:  {eval_losses[0]:.4f} → {eval_losses[-1]:.4f}')

In [ ]:
# Cell 11: Comprehensive evaluation on held-out set
import torch

SYSTEM_PROMPT = "You are VendorCenter's AI assistant. Given the user's message, produce a JSON decision with: intent (SERVICE_SEARCH | FAQ | GREETING | MY_BOOKINGS | AVAILABLE_SERVICES | COMPLAINT | RESCHEDULE | CANCEL_BOOKING | REFUND | VENDOR_INFO | LOCATION | UNKNOWN), action (SHOW_RESULTS | ASK_LOCATION | ASK_DETAILS | NAVIGATE | SHOW_MY_BOOKINGS | SHOW_CATEGORIES | NONE), service (category name or empty), mode (SERVICE | CHAT), confidence (0-1), and message (natural language response). Support English, Hinglish, and Marathi."

def run_inference(user_msg):
    """Run model inference and return raw text."""
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        + SYSTEM_PROMPT + '<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        + user_msg + '<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.inference_mode():
        output = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )
    result = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    del inputs, output
    return result

# Evaluate on eval set
json_valid = 0
intent_correct = 0
total = len(eval_formatted)
intent_results = {}

print(f'Evaluating on {total} held-out examples...')
print(f'(Uncached generation to avoid unsloth rotary bug — ~20-30 min on T4)\n')
for i, ex in enumerate(eval_formatted):
    expected_intent = ex['intent']
    response = run_inference(ex['raw_input'])

    if expected_intent not in intent_results:
        intent_results[expected_intent] = [0, 0]
    intent_results[expected_intent][1] += 1

    try:
        parsed = json.loads(response)
        json_valid += 1
        if parsed.get('intent') == expected_intent:
            intent_correct += 1
            intent_results[expected_intent][0] += 1
    except json.JSONDecodeError:
        pass

    if (i + 1) % 10 == 0:
        print(f'  Processed {i+1}/{total} — JSON: {json_valid}/{i+1}, Intent: {intent_correct}/{i+1}')

print(f'\n{"="*50}')
print(f'EVALUATION RESULTS ({total} examples)')
print(f'{"="*50}')
print(f'JSON validity:    {json_valid}/{total} ({json_valid/total*100:.1f}%)')
print(f'Intent accuracy:  {intent_correct}/{total} ({intent_correct/total*100:.1f}%)')
print(f'\nPer-intent accuracy:')
for intent in sorted(intent_results.keys()):
    correct, count = intent_results[intent]
    acc = correct / count * 100 if count > 0 else 0
    status = '✓' if acc >= 80 else '⚠' if acc >= 50 else '✗'
    print(f'  {status} {intent:25s} {correct:3d}/{count:3d} ({acc:.0f}%)')

In [ ]:
# Cell 12: Multi-intent test suite (manual spot-checks)
test_cases = [
    ('I need a plumber near me', 'SERVICE_SEARCH'),
    ('How do I book a service?', 'FAQ'),
    ('hello', 'GREETING'),
    ('Show my bookings', 'MY_BOOKINGS'),
    ('What services are available?', 'AVAILABLE_SERVICES'),
    ('The vendor did a terrible job', 'COMPLAINT'),
    ('I need to reschedule my booking', 'RESCHEDULE'),
    ('Cancel my booking', 'CANCEL_BOOKING'),
    ('Where is my refund?', 'REFUND'),
    ('Show me vendor reviews', 'VENDOR_INFO'),
    ('I am in Pune', 'LOCATION'),
    ('Tell me a joke', 'UNKNOWN'),
    ('AC kharab ho gaya hai', 'SERVICE_SEARCH'),
    ('mujhe cancel karna hai', 'CANCEL_BOOKING'),
    ('complaint raise karna hai', 'COMPLAINT'),
    ('booking ka status kya hai?', 'MY_BOOKINGS'),
    ('मला प्लंबर हवा', 'SERVICE_SEARCH'),
    ('URGENT pipe burst help!!!!', 'SERVICE_SEARCH'),
    ('worst service ever', 'COMPLAINT'),
    ('need electrician asap', 'SERVICE_SEARCH'),
]

print(f'Running {len(test_cases)} test cases...\n')
passed = 0
for user_msg, expected_intent in test_cases:
    response = run_inference(user_msg)
    try:
        parsed = json.loads(response)
        actual_intent = parsed.get('intent', 'PARSE_ERROR')
        match = actual_intent == expected_intent
        if match:
            passed += 1
        icon = '✓' if match else '✗'
        print(f'{icon} "{user_msg[:45]:<45s}" expected={expected_intent:<20s} got={actual_intent}')
    except json.JSONDecodeError:
        print(f'✗ "{user_msg[:45]:<45s}" expected={expected_intent:<20s} got=INVALID_JSON')

print(f'\nPassed: {passed}/{len(test_cases)} ({passed/len(test_cases)*100:.0f}%)')

In [ ]:
# Cell 13: Save LoRA adapter (checkpoint)
model.save_pretrained('vendorcenter-lora-final')
tokenizer.save_pretrained('vendorcenter-lora-final')
print('LoRA adapter saved to vendorcenter-lora-final/')

In [ ]:
# Cell 14: Merge LoRA into base model and export GGUF Q4_K_M
model.save_pretrained_gguf(
    'vendorcenter-gguf',
    tokenizer,
    quantization_method='q4_k_m',
)

import os
gguf_files = [f for f in os.listdir('vendorcenter-gguf') if f.endswith('.gguf')]
for f in gguf_files:
    size_mb = os.path.getsize(f'vendorcenter-gguf/{f}') / 1024 / 1024
    print(f'{f}: {size_mb:.0f} MB')

In [ ]:
# Cell 15: Push GGUF to HuggingFace Hub
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1: Get your HF token from https://huggingface.co/settings/tokens
#         → Click "New token" → Select "Write" permission → Copy token
# STEP 2: Paste token below and set your username
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

HF_TOKEN = ''  # <-- Paste your HuggingFace WRITE token here (starts with hf_)
HF_REPO = 'timesprimeaj/vendorcenter-assistant-3b-gguf'  # <-- Your HF username/repo

if not HF_TOKEN:
    print('⚠️  Set HF_TOKEN above first!')
    print('   1. Go to: https://huggingface.co/settings/tokens')
    print('   2. Click "New token" → Name: "vendorcenter" → Type: Write')
    print('   3. Copy the token and paste it in HF_TOKEN above')
    print('   4. Re-run this cell')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

    # Create repo if it doesn't exist
    api.create_repo(HF_REPO, exist_ok=True, repo_type='model')
    print(f'Repo ready: https://huggingface.co/{HF_REPO}')

    # Upload all GGUF files
    import os
    gguf_files = [f for f in os.listdir('vendorcenter-gguf') if f.endswith('.gguf')]
    for f in gguf_files:
        fpath = f'vendorcenter-gguf/{f}'
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f'Uploading {f} ({size_mb:.0f} MB)...')
        api.upload_file(
            path_or_fileobj=fpath,
            path_in_repo=f,
            repo_id=HF_REPO,
        )
        print(f'  ✓ Uploaded')

    print(f'\n✅ Done! Model at: https://huggingface.co/{HF_REPO}')

In [ ]:
# Cell 16: Download GGUF locally (alternative to Hub push)
from google.colab import files
import os

gguf_files = [f for f in os.listdir('vendorcenter-gguf') if f.endswith('.gguf')]
for f in gguf_files:
    fpath = f'vendorcenter-gguf/{f}'
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f'Downloading {f} ({size_mb:.0f} MB)...')
    files.download(fpath)

print('Download complete! Upload this GGUF manually to HF if needed.')

## Next Steps

1. **Push GGUF to HuggingFace** (Cell 15) or **download locally** (Cell 16)
2. **Deploy on HF Spaces**: Use the `model/hf-space/Dockerfile` from the repo
3. **Set backend env**: `SELF_HOSTED_LLM_URL=https://YOUR-SPACE.hf.space` on Railway
4. **Test**: The provider chain will now try your model first → Groq → Gemini → Static

### Improving the Model
- Run `npx tsx src/scripts/generate-training-data.ts` to regenerate data after adding categories
- Add real user queries from `ai_query_logs` table to training data
- If eval loss plateaus but intent accuracy is low: increase LoRA rank (64→128) or epochs (3→5)
- If JSON validity < 90%: add more diverse output format examples
- Monitor per-intent accuracy — low scores indicate need for more examples in that intent
- Try `unsloth/Llama-3.2-1B-Instruct` for faster/smaller model (if accuracy holds)